# GemAI — RAG Evaluation v2

### Objective

Re-evaluate GemAI **as the app actually runs today**, and test whether the structure-aware chunker in `lib/seed.ts` beats the plain character splitter.

`rag-evaluation.ipynb` (v1) chose the prose chunk size (800/100) and `topK` (8). Since then the app has changed in ways v1 doesn't measure:

| | v1 | App now |
|---|---|---|
| Chunking | LangChain `RecursiveCharacterTextSplitter(800, 100)` | Structure-aware: one chunk per USGS fact-sheet entry, labelled Smithsonian catalog rows, sentence-aligned prose that can cross pages, caption/boilerplate cleanup |
| Retrieval | k = 4 baseline | k = 8 |
| Prompt | Context-stuffing prompt | Tool call returning numbered passages, inline `[n]` citations, small-talk and off-topic rules |
| Groundedness | Judged without seeing the retrieved passages (so it measured general factual accuracy) | — |

### What this notebook does

1. Loads the **app's own chunks** (from `lib/seed.ts --dry-run`) and, for comparison, the old splitter's chunks.
2. Builds an 18-question dataset: v1's 10 questions (three references corrected against the PDFs) plus questions aimed at the fact sheet, the catalog, tables, and two questions the corpus can't answer.
3. Reproduces the app's answer step: the **system prompt, tool description, titles and k are read from `app/api/chat/route.ts`**, and passages are passed back as a tool result, as in the app.
4. Scores five metrics in LangSmith: correctness, helpfulness, groundedness **against the retrieved passages**, citation accuracy, and retrieval hit rate.
5. Compares old vs new chunking at k = 8, and the new chunking at k = 4.
6. Tests the tool-call decision (search or not) against the running app.

### Running it

- `.env.local` at the repo root needs `OPENAI_API_KEY` and `LANGCHAIN_API_KEY` (anything missing is asked for).
- `npm install` (Part 1 runs the TypeScript chunker).
- Part 6 needs the app running: `npm run dev` (or set `APP_URL`).

## Part 0: Setup

In [1]:
import json
import os
import re
import subprocess
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO / ".env.local")

for key in ["OPENAI_API_KEY", "LANGCHAIN_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ.setdefault("LANGCHAIN_PROJECT", "GemAI-Eval-v2")
print(f"Repo: {REPO}")
print(f"LangSmith project: {os.environ['LANGCHAIN_PROJECT']}")

Repo: /home/celine/AIM/notebooks/graded-mini-projects/14
LangSmith project: GemAI-Eval-v2


### Read the app's configuration

The prompt, tool description, document titles and `topK` come straight from the route handler, so this notebook can't drift from the app.

In [2]:
route = (REPO / "app/api/chat/route.ts").read_text()

SYSTEM_PROMPT = re.search(r"system:\s*`([\s\S]*?)`", route).group(1)
TOOL_DESCRIPTION = re.search(r"getInformation: tool\(\{\s*description:\s*'([^']+)'", route).group(1)
SOURCE_TITLES = dict(re.findall(r"'([\w.-]+\.pdf)':\s*'([^']+)'", route))
APP_TOP_K = int(re.search(r"topK:\s*(\d+)", route).group(1))
APP_MODEL = re.search(r"openai\('([\w.-]+)'\)", route).group(1)
APP_EMBEDDING = re.search(r"openai\.embedding\('([\w.-]+)'\)", route).group(1)
APP_MAX_TOKENS = int(re.search(r"maxTokens:\s*(\d+)", route).group(1))

print(f"Model: {APP_MODEL} | Embedding: {APP_EMBEDDING} | topK: {APP_TOP_K} | maxTokens: {APP_MAX_TOKENS}")
print(f"Titles: {SOURCE_TITLES}")
print(f"\nSystem prompt ({len(SYSTEM_PROMPT)} chars):\n{SYSTEM_PROMPT[:400]}…")

Model: gpt-4o-mini | Embedding: text-embedding-3-small | topK: 8 | maxTokens: 1024
Titles: {'usgs-natural-gemstones.pdf': 'USGS — Natural Gemstones', 'usgs-gemstones-mcs-2026.pdf': 'USGS — Mineral Commodity Summaries 2026: Gemstones', 'smithsonian-gems.pdf': 'Smithsonian — Gems in the Smithsonian Institution'}

System prompt (1902 chars):
You are GemAI, a friendly and knowledgeable assistant specializing in gemstones.
Your knowledge comes from USGS gemstone publications and the Smithsonian Institution gem collection reference.

Decide which kind of message you received:
1. Greetings, thanks, small talk, or questions about what you can do: reply briefly and warmly WITHOUT calling any tool, and suggest a few things to ask about (e.g.…


## Part 1: Chunks

- **A — old splitter:** PyMuPDF pages → `RecursiveCharacterTextSplitter(800, 100)`, as in v1.
- **B — app chunker:** exactly what `npm run seed` indexes, via `npx tsx lib/seed.ts --dry-run`.

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# B: the app's chunks
proc = subprocess.run(["npx", "tsx", "lib/seed.ts", "--dry-run"], cwd=REPO, capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr or proc.stdout)
app_chunks = json.loads((REPO / "chunks.dry-run.json").read_text())
docs_b = [
    Document(page_content=c["text"], metadata={"source": c["source"], "page": c["page"], "pageEnd": c["pageEnd"]})
    for c in app_chunks
]

# A: the old splitter
pages = []
for pdf in sorted((REPO / "data").glob("*.pdf")):
    for d in PyMuPDFLoader(str(pdf)).load():
        page = d.metadata["page"] + 1  # PyMuPDF pages are 0-based
        pages.append(Document(page_content=d.page_content, metadata={"source": pdf.name, "page": page, "pageEnd": page}))
docs_a = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(pages)

print(f"A (old splitter): {len(docs_a)} chunks")
print(f"B (app chunker):  {len(docs_b)} chunks")

A (old splitter): 205 chunks
B (app chunker):  204 chunks


/home/celine/AIM/notebooks/graded-mini-projects/14/.venv/lib/python3.11/site-packages/langchain_community/document_loaders/parsers/pdf.py:322: UserWarning: Warning: Empty content on page 19 of document /home/celine/AIM/notebooks/graded-mini-projects/14/data/usgs-natural-gemstones.pdf
  warnings.warn(


Same content, chunked both ways: the Smithsonian catalog row for the Hope Diamond.

In [4]:
def show(docs, needle, label):
    d = next(d for d in docs if needle in d.page_content)
    i = d.page_content.index(needle)
    print(f"--- {label} ({d.metadata['source']} p.{d.metadata['page']}) ---")
    print(d.page_content[max(0, i - 250): i + 150], "\n")

show(docs_a, "(The Hope)", "A")
show(docs_b, "(The Hope)", "B")

--- A (smithsonian-gems.pdf p.32) ---
iption, if any;
place of origin; and U. S. National Museum catalog number and name of
donor. Gems in the Lea and Roebling collections usually are indicated by
the letters “L” and “R.”
DIAMOND
127, colorless (The Portuguese), Brazil (3398)
44.5, blue (The Hope), India (3551, Winston)
18.3, yellow (The Shephard), South Africa (3406)
2.9, pink, Tanzania (3772, De Young)
CORUNDUM: Ruby
50, red-violet  

--- B (smithsonian-gems.pdf p.32) ---
Smithsonian collection list — DIAMOND (weight in carats, color, origin (catalog number, donor or collection: L = Lea, R = Roebling))
127, colorless (The Portuguese), Brazil (3398)
44.5, blue (The Hope), India (3551, Winston)
18.3, yellow (The Shephard), South Africa (3406)
2.9, pink, Tanzania (3772, De Young) 



## Part 2: Evaluation dataset

18 questions. Every reference was checked against the PDF text, and `expected` lists where the answer is (file, page) for the retrieval metric.

Changes to v1's 10 questions:
- **Hope Diamond:** reference now includes 44.5 carats, blue, India (Smithsonian collection list, p.32).
- **Gem cuts:** v1 listed "tumbled stones" as a cut; the source says tumbled stones are only smoothed. Brilliant and step cuts are the commonest; cabochons for opaque and star stones.
- **Pegmatites:** v1's reference came from the *stream gravels* definition and implied emeralds come from pegmatites; the Smithsonian text says the finest emeralds do not.
- **Specific gravity, synthetics:** wording aligned with the source.

New questions target structures the chunker handles specially (fact sheet, catalog, birthstone table, the commodity summary) and two questions the corpus does not answer (`expected` empty).

In [5]:
import hashlib

import pandas as pd

NG = "usgs-natural-gemstones.pdf"
SG = "smithsonian-gems.pdf"
MCS = "usgs-gemstones-mcs-2026.pdf"

test_data = [
    # --- v1 questions ---
    {"category": "v1", "question": "What is the hardness of diamond on the Mohs scale?",
     "reference": "Diamond has a hardness of 10 on the Mohs scale, the highest on the scale; it is the hardest substance known.",
     "expected": [[NG, 7], [SG, 12]]},
    {"category": "v1", "question": "Where were sapphires first found in Montana?",
     "reference": "Sapphires in Judith Basin County, Montana, were first found when the gravels were worked for gold from 1895 to 1930.",
     "expected": [[NG, 4]]},
    {"category": "v1", "question": "What is the Hope Diamond?",
     "reference": "The Hope Diamond is a 44.5-carat blue diamond from India in the Smithsonian's National Gem Collection, given by Harry Winston, Inc.",
     "expected": [[SG, 32], [SG, 2]]},
    {"category": "v1", "question": "What makes a mineral qualify as a gemstone?",
     "reference": "A gemstone is a mineral, stone, or organic matter that can be cut and polished for jewelry or ornament. A precious gemstone has beauty, durability, and rarity; a semiprecious gemstone has only one or two of these qualities.",
     "expected": [[NG, 4]]},
    {"category": "v1", "question": "What is specific gravity and how is it used in gemology?",
     "reference": "Specific gravity is the number of times heavier a gemstone is than an equal volume of water (the ratio of its density to water's). Each gem species has its own specific gravity, which can be measured without harming the stone, so it is a valuable aid in identification.",
     "expected": [[NG, 7], [SG, 4]]},
    {"category": "v1", "question": "How did the Smithsonian gem collection begin?",
     "reference": "The National Gem Collection began in 1884 when Prof. F. W. Clarke, honorary curator of the Division of Mineralogy, prepared an exhibit of American precious stones for the Smithsonian's display at the New Orleans Exposition.",
     "expected": [[SG, 2]]},
    {"category": "v1", "question": "What are the main types of gem cuts?",
     "reference": "Fine gem rough is mainly faceted. The brilliant and step cuts are by far the commonest basic cuts, with fancy cuts such as the baguette also used. Opaque stones and those showing stars or cat's-eyes are cut as cabochons, a raised and rounded form.",
     "expected": [[SG, 6], [SG, 7]]},
    {"category": "v1", "question": "What is the difference between a natural and synthetic gemstone?",
     "reference": "A synthetic gemstone is made in the laboratory and, to qualify as synthetic, must be identical chemically and structurally with its natural counterpart, which formed in nature. Synthetic sapphire, ruby, spinel, emerald, and rutile are produced commercially.",
     "expected": [[SG, 9]]},
    {"category": "v1", "question": "What gemstones are found in pegmatite deposits?",
     "reference": "Pegmatites are very coarse-grained igneous (granite) rocks. Beryl is usually found in pegmatites, sometimes as very large crystals, but the finest emeralds are not found in pegmatites.",
     "expected": [[SG, 17], [NG, 4]]},
    {"category": "v1", "question": "What was the combined value of US gemstone production in 2025?",
     "reference": "The combined value of U.S. natural and synthetic gemstone output in 2025 was an estimated $47 million, a decrease of 30% compared with 2024.",
     "expected": [[MCS, 1]]},
    # --- fact sheet ---
    {"category": "fact sheet", "question": "What is the hardness of feldspar?",
     "reference": "Feldspar has a hardness of 6 to 6.5 on the Mohs scale.",
     "expected": [[NG, 7]]},
    {"category": "fact sheet", "question": "What colors does spinel come in?",
     "reference": "Spinel varieties include balas ruby (red), almandine spinel (purple red), rubicelle (orange), sapphire spinel and gahnospinel (blue), and chlorspinel (green).",
     "expected": [[NG, 8], [NG, 9], [SG, 21]]},
    # --- catalog ---
    {"category": "catalog", "question": "How many carats is the Hope Diamond?",
     "reference": "The Hope Diamond weighs 44.5 carats.",
     "expected": [[SG, 32]]},
    {"category": "catalog", "question": "What is the largest topaz in the Smithsonian collection list?",
     "reference": "The largest topaz listed is a 7725-carat yellow topaz from Brazil.",
     "expected": [[SG, 33]]},
    # --- tables ---
    {"category": "table", "question": "What is the birthstone for October?",
     "reference": "The birthstone for October is opal or tourmaline.",
     "expected": [[NG, 10], [SG, 11]]},
    {"category": "table", "question": "Where does the US import most of its diamonds from?",
     "reference": "From 2021 to 2024, by value, U.S. diamond imports came from India (46%), Israel (27%), Belgium (11%), South Africa (4%), and other countries (12%).",
     "expected": [[MCS, 1], [MCS, 2]]},
    # --- not in the corpus ---
    {"category": "unanswerable", "question": "How much does a carat of tanzanite cost?",
     "reference": "The sources do not contain information about tanzanite or its price.",
     "expected": []},
    {"category": "unanswerable", "question": "What is the hardness of moissanite?",
     "reference": "The sources do not contain information about moissanite.",
     "expected": []},
]

df = pd.DataFrame(test_data)
print(f"{len(df)} questions:", df["category"].value_counts().to_dict())
df[["category", "question", "reference"]]

18 questions: {'v1': 10, 'fact sheet': 2, 'catalog': 2, 'table': 2, 'unanswerable': 2}


,category,question,reference
0,v1,What is the hardness of diamond on the Mohs sc...,Diamond has a hardness of 10 on the Mohs scale...
1,v1,Where were sapphires first found in Montana?,"Sapphires in Judith Basin County, Montana, wer..."
2,v1,What is the Hope Diamond?,The Hope Diamond is a 44.5-carat blue diamond ...
3,v1,What makes a mineral qualify as a gemstone?,"A gemstone is a mineral, stone, or organic mat..."
4,v1,What is specific gravity and how is it used in...,Specific gravity is the number of times heavie...
5,v1,How did the Smithsonian gem collection begin?,The National Gem Collection began in 1884 when...
6,v1,What are the main types of gem cuts?,Fine gem rough is mainly faceted. The brillian...
7,v1,What is the difference between a natural and s...,A synthetic gemstone is made in the laboratory...
8,v1,What gemstones are found in pegmatite deposits?,Pegmatites are very coarse-grained igneous (gr...
9,v1,What was the combined value of US gemstone pro...,The combined value of U.S. natural and synthet...


In [6]:
from langsmith import Client

client = Client()

# The dataset name includes a hash of its contents, so editing test_data creates a new
# dataset instead of silently reusing a stale one, and re-running reuses the same one.
digest = hashlib.sha1(json.dumps(test_data, sort_keys=True).encode()).hexdigest()[:8]
DATASET_NAME = f"GemAI Eval v2 ({digest})"

if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Reusing dataset: {DATASET_NAME}")
else:
    dataset = client.create_dataset(DATASET_NAME, description="GemAI v2: app chunker vs old splitter")
    client.create_examples(
        inputs=[{"question": t["question"]} for t in test_data],
        outputs=[{"answer": t["reference"], "expected": t["expected"]} for t in test_data],
        metadata=[{"category": t["category"]} for t in test_data],
        dataset_id=dataset.id,
    )
    print(f"Created dataset: {DATASET_NAME}")

Reusing dataset: GemAI Eval v2 (eef67a20)


## Part 3: Pipeline that mirrors the app

In the app, the model calls `getInformation`, the tool returns numbered passages (`id`, `title`, `page`, `pageEnd`, `text`), and the model answers from them. Here the same messages are built explicitly: system prompt → question → tool call → tool result → answer.

Two differences from the app, both deliberate:
- The search query is the question itself; in the app the model writes its own query (usually a close paraphrase).
- Retrieval always happens. Whether the app *decides* to search is tested separately in Part 6.

In [7]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

embeddings = OpenAIEmbeddings(model=APP_EMBEDDING)

stores = {
    "A": QdrantVectorStore.from_documents(docs_a, embeddings, location=":memory:", collection_name="old-splitter"),
    "B": QdrantVectorStore.from_documents(docs_b, embeddings, location=":memory:", collection_name="app-chunker"),
}

GET_INFORMATION = {
    "type": "function",
    "function": {
        "name": "getInformation",
        "description": TOOL_DESCRIPTION,
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "the topic, term, or sub-question to search for"}},
            "required": ["query"],
        },
    },
}
# tool_choice="none": the passages are already supplied, so the model must answer.
answer_llm = ChatOpenAI(model=APP_MODEL, max_tokens=APP_MAX_TOKENS).bind_tools([GET_INFORMATION], tool_choice="none")


def to_passages(docs):
    return [
        {
            "id": i + 1,
            "text": d.page_content,
            "page": d.metadata["page"],
            "pageEnd": d.metadata["pageEnd"],
            "source": d.metadata["source"],
            "title": SOURCE_TITLES.get(d.metadata["source"], d.metadata["source"]),
        }
        for i, d in enumerate(docs)
    ]


def make_target(store, k):
    def target(inputs: dict) -> dict:
        question = inputs["question"]
        passages = to_passages(store.similarity_search(question, k=k))
        messages = [
            SystemMessage(SYSTEM_PROMPT),
            HumanMessage(question),
            AIMessage("", tool_calls=[{"name": "getInformation", "args": {"query": question}, "id": "call_1"}]),
            ToolMessage(json.dumps(passages), tool_call_id="call_1"),
        ]
        return {"output": answer_llm.invoke(messages).content, "contexts": passages}

    return target


print(make_target(stores["B"], APP_TOP_K)({"question": "What is the hardness of feldspar?"})["output"])

Feldspar has a hardness of 6 to 6.5 on the Mohs scale [2]. This means it can scratch minerals with a lower hardness, such as quartz (which has a hardness of 7) and calcite (hardness of 3) [1].


## Part 4: Evaluators

| Metric | Judges | Change from v1 |
|---|---|---|
| `correctness` | Answer vs reference answer | Same (`qa`) |
| `helpfulness` | Helpfulness given the reference | Same |
| `groundedness` | Is every claim supported by **the retrieved passages**? | Fixed: v1 never showed the judge the passages |
| `citation_accuracy` | Share of cited sentences whose cited passage actually supports them | New |
| `retrieval_hit` | Is a passage from an expected page in the top k? (no LLM) | New |

For unanswerable questions, `citation_accuracy` and `retrieval_hit` are skipped; `correctness` checks the bot says the sources don't cover it.

In [8]:
from langsmith.evaluation import LangChainStringEvaluator
from pydantic import BaseModel, Field

eval_llm = ChatOpenAI(model="gpt-4.1", temperature=0)


def passages_text(passages):
    return "\n\n".join(f"[{p['id']}] ({p['title']}, p.{p['page']}) {p['text']}" for p in passages)


def answer_fields(run, example):
    return {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }


correctness = LangChainStringEvaluator("qa", config={"llm": eval_llm}, prepare_data=answer_fields)

helpfulness = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {"helpfulness": "Is this submission helpful to the user, taking into account the correct reference answer?"},
        "llm": eval_llm,
    },
    prepare_data=answer_fields,
)

groundedness = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "groundedness": (
                "Is every factual claim in the submission supported by the reference passages? "
                "A claim that is true but not stated in the passages counts as unsupported. "
                "Saying that the passages do not cover something is not a factual claim."
            )
        },
        "llm": eval_llm,
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": passages_text(run.outputs["contexts"]),
        "input": example.inputs["question"],
    },
)


class CitationCheck(BaseModel):
    cited_sentences: int = Field(description="Number of sentences or list items that carry at least one [n] citation")
    supported: int = Field(description="How many of those are fully supported by the passage(s) they cite")
    unsupported_examples: list[str] = Field(description="Up to three cited statements their passages do not support")


citation_judge = eval_llm.with_structured_output(CitationCheck)


def citation_accuracy(run, example):
    if not example.outputs["expected"]:
        return {"key": "citation_accuracy", "score": None, "comment": "unanswerable question"}
    answer = run.outputs["output"]
    if not re.search(r"\[\d+", answer):
        return {"key": "citation_accuracy", "score": 0, "comment": "answer has no citations"}
    check = citation_judge.invoke(
        "Check the inline citations in an answer. For each sentence or list item that carries one or more [n] "
        "markers, decide whether the passages it cites support it. Judge only against the cited passages, not "
        f"general knowledge.\n\nPASSAGES:\n{passages_text(run.outputs['contexts'])}\n\nANSWER:\n{answer}"
    )
    if check.cited_sentences == 0:
        return {"key": "citation_accuracy", "score": 0, "comment": "no cited sentences found"}
    return {
        "key": "citation_accuracy",
        "score": min(check.supported, check.cited_sentences) / check.cited_sentences,
        "comment": "; ".join(check.unsupported_examples),
    }


def retrieval_hit(run, example):
    expected = example.outputs["expected"]
    if not expected:
        return {"key": "retrieval_hit", "score": None, "comment": "unanswerable question"}
    hit = any(
        p["source"] == source and p["page"] <= page <= p["pageEnd"]
        for p in run.outputs["contexts"]
        for source, page in expected
    )
    return {"key": "retrieval_hit", "score": int(hit)}


evaluators = [correctness, helpfulness, groundedness, citation_accuracy, retrieval_hit]
print("Evaluators: correctness, helpfulness, groundedness, citation_accuracy, retrieval_hit")

Evaluators: correctness, helpfulness, groundedness, citation_accuracy, retrieval_hit


## Part 5: Experiments

All three use the same dataset, model, prompt and evaluators.

| Experiment | Chunks | k |
|---|---|---|
| `A-old-splitter-k8` | Old splitter | 8 |
| `B-app-chunker-k8` | App chunker (what the app runs) | 8 |
| `B-app-chunker-k4` | App chunker | 4 |

**Hypothesis:** B beats A mainly on fact-sheet and catalog questions (retrieval hit, correctness), because their facts are no longer split from their context; B should also be at least as grounded. If B at k=4 matches B at k=8, the larger k was compensating for poor chunks.

In [9]:
from langsmith.evaluation import evaluate

EXPERIMENTS = {
    "A-old-splitter-k8": (stores["A"], 8),
    "B-app-chunker-k8": (stores["B"], 8),
    "B-app-chunker-k4": (stores["B"], 4),
}

results = {}
for name, (store, k) in EXPERIMENTS.items():
    print(f"\n=== {name} ===")
    results[name] = evaluate(
        make_target(store, k),
        data=DATASET_NAME,
        evaluators=evaluators,
        experiment_prefix=name,
        metadata={"chunks": name.split("-k")[0], "k": k, "model": APP_MODEL},
        max_concurrency=4,
    )


=== A-old-splitter-k8 ===


/home/celine/AIM/notebooks/graded-mini-projects/14/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'A-old-splitter-k8-dd605826' at:
https://smith.langchain.com/o/52420107-7b88-4243-acba-2f92a36f9bc4/datasets/28fdd709-594f-4fac-9aff-79c7585acb31/compare?selectedSessions=e45d8e50-2595-4f36-b882-14d84d25e7fe




0it [00:00, ?it/s]

1it [00:06,  6.21s/it]

...

15it [00:29,  1.91s/it]

16it [00:31,  1.67s/it]

17it [00:32,  1.71s/it]

18it [00:34,  1.63s/it]

18it [00:34,  1.90s/it]

### Results

In [10]:
METRICS = ["correctness", "helpfulness", "groundedness", "citation_accuracy", "retrieval_hit"]
category_by_question = {t["question"]: t["category"] for t in test_data}


def to_frame(name, res):
    df = res.to_pandas()
    df["question"] = df["inputs.question"]
    df["category"] = df["question"].map(category_by_question)
    df["experiment"] = name
    return df.rename(columns={f"feedback.{m}": m for m in METRICS})


rows = pd.concat([to_frame(name, res) for name, res in results.items()], ignore_index=True)
for m in METRICS:
    rows[m] = pd.to_numeric(rows[m], errors="coerce")

summary = rows.groupby("experiment", sort=False)[METRICS].mean().round(2)
summary

,correctness,helpfulness,groundedness,citation_accuracy,retrieval_hit
experiment,,,,,
A-old-splitter-k8,0.89,0.83,0.83,0.92,0.94
B-app-chunker-k8,0.94,1.00,0.83,0.96,1.00
B-app-chunker-k4,0.89,0.94,0.67,0.98,1.00


By question type (correctness / retrieval hit):

In [11]:
by_category = (
    rows.groupby(["category", "experiment"], sort=False)[["correctness", "retrieval_hit", "groundedness"]]
    .mean()
    .round(2)
    .unstack("experiment")
)
by_category

correctness                                    \
experiment   A-old-splitter-k8 B-app-chunker-k8 B-app-chunker-k4   
category                                                           
table                      1.0              1.0              0.5   
unanswerable               0.5              0.5              0.5   
catalog                    0.5              1.0              1.0   
fact sheet                 1.0              1.0              1.0   
v1                         1.0              1.0              1.0   

                 retrieval_hit                                    \
experiment   A-old-splitter-k8 B-app-chunker-k8 B-app-chunker-k4   
category                                                           
table                      1.0              1.0              1.0   
unanswerable               NaN              NaN              NaN   
catalog                    0.5              1.0              1.0   
fact sheet                 1.0              1.0              1.0   
v1                         1.0              1.0              1.0   

                  groundedness                                    
experiment   A-old-splitter-k8 B-app-chunker-k8 B-app-chunker-k4  
category                                                          
table                      1.0              1.0              0.5  
unanswerable               0.5              0.5              0.5  
catalog                    1.0              1.0              1.0  
fact sheet                 1.0              1.0              0.5  
v1                         0.8              0.8              0.7

Answers that lost points, for inspection:

In [12]:
pd.set_option("display.max_colwidth", 200)
cols = ["experiment", "category", "question"] + METRICS + ["outputs.output"]
misses = rows[(rows["correctness"] < 1) | (rows["groundedness"] < 1) | (rows["citation_accuracy"] < 1)]
misses[cols].sort_values(["question", "experiment"])

,experiment,category,question,correctness,helpfulness,groundedness,citation_accuracy,retrieval_hit,outputs.output
13,A-old-splitter-k8,v1,What are the main types of gem cuts?,1,1,0,1.000000,1.0,"The main types of gem cuts include:\n\n1. **Brilliant Cut**: This is the most common cut used for diamonds, designed to maximize internal reflection and brilliance [2].\n\n2. **Step Cut**: Another..."
29,B-app-chunker-k8,v1,What are the main types of gem cuts?,1,1,0,0.750000,1.0,"The main types of gem cuts include:\n\n1. **Brilliant Cut**: This is the most commonly used cut, designed to maximize the reflection of light within the gemstone, resulting in high brilliance and ..."
43,B-app-chunker-k4,fact sheet,What colors does spinel come in?,1,1,0,1.000000,1.0,"Spinel comes in a variety of colors, including:\n\n- **Red**: Balas ruby and Almandine spinel\n- **Purple Red**: Almandine spinel\n- **Orange**: Rubicelle\n- **Blue**: Sapphire spinel\n- **Green**..."
25,B-app-chunker-k8,fact sheet,What colors does spinel come in?,1,1,1,0.888889,1.0,"Spinel comes in a variety of colors, including:\n\n- Purplish red (Almandine spinel)\n- Orange-red (Rubicelle)\n- Rose red (Balas ruby)\n- Deep red (Ruby spinel)\n- Translucent grass green (Chloro..."
11,A-old-splitter-k8,v1,What gemstones are found in pegmatite deposits?,1,0,0,0.800000,1.0,"Gemstones found in pegmatite deposits include:\n\n1. **Beryl** - This mineral is commonly found in pegmatites and can form large crystals, although many may not be of gem quality due to shattering..."
51,B-app-chunker-k4,v1,What is the Hope Diamond?,1,1,0,0.666667,1.0,"The Hope Diamond is a famous blue diamond, weighing approximately 44.5 carats, and it is part of the Smithsonian Institution's National Gem Collection. Its origin is traced back to India, and it w..."
32,B-app-chunker-k8,v1,What is the Hope Diamond?,1,1,1,0.666667,1.0,"The Hope Diamond is a blue diamond weighing 44.5 carats, believed to have originated from India [2]. It is noted for its remarkable color and history, and it is part of the National Gem Collection..."
37,B-app-chunker-k4,table,What is the birthstone for October?,1,1,0,1.000000,1.0,"The birthstones for October are Opal and Tourmaline [1][2]. Opal is known for its unique play-of-color, while Tourmaline comes in a variety of colors, making it a versatile choice. If you have mor..."
45,B-app-chunker-k4,v1,What is the difference between a natural and synthetic gemstone?,1,1,0,1.000000,1.0,"The primary difference between natural and synthetic gemstones lies in their origin. Natural gemstones are formed through geological processes over millions of years, while synthetic gemstones are..."
1,A-old-splitter-k8,unanswerable,What is the hardness of moissanite?,0,1,0,NaN,NaN,"The hardness of moissanite is not specifically mentioned in the sources I accessed. However, it is commonly known to be very close to that of diamond, typically rated around 9.25 on the Mohs scale..."


## Part 6: Does the app decide correctly when to search?

The pipeline above always retrieves. In the app the model chooses whether to call `getInformation`, so this part sends messages to the running app's `/api/chat` and checks the stream for a tool call (lines starting with `9:`). Each case runs twice; the pause keeps under the app's rate limit (10 requests/minute).

In [13]:
import time

import requests

APP_URL = os.environ.get("APP_URL", "http://localhost:3000")

SAPPHIRE_TURN = [
    {"role": "user", "content": "What color is sapphire?"},
    {"role": "assistant", "content": "Sapphires are most commonly blue."},
]
cases = [
    ("hi!", [{"role": "user", "content": "hi!"}], False),
    ("what can you do?", [{"role": "user", "content": "what can you do?"}], False),
    ("thanks (follow-up)", SAPPHIRE_TURN + [{"role": "user", "content": "thanks, that helps!"}], False),
    ("How do I cook carbonara?", [{"role": "user", "content": "How do I cook carbonara?"}], False),
    ("What color is sapphire?", [{"role": "user", "content": "What color is sapphire?"}], True),
    ("is my stone good?", [{"role": "user", "content": "is my stone good?"}], True),
]


def ask_app(messages):
    stream = requests.post(f"{APP_URL}/api/chat", json={"messages": messages}, timeout=90).text.splitlines()
    searched = any(line.startswith("9:") for line in stream)
    text = "".join(json.loads(line[2:]) for line in stream if line.startswith("0:"))
    return searched, text


try:
    requests.get(APP_URL, timeout=5)
except requests.RequestException:
    print(f"App not reachable at {APP_URL}; start it with `npm run dev` and re-run this cell.")
else:
    tool_rows = []
    for label, messages, should_search in cases:
        for attempt in (1, 2):
            searched, text = ask_app(messages)
            tool_rows.append({"message": label, "attempt": attempt, "should_search": should_search,
                              "searched": searched, "correct": searched == should_search, "reply": text[:120]})
            time.sleep(7)
    tool_df = pd.DataFrame(tool_rows)
    print(f"Correct search decisions: {tool_df['correct'].sum()}/{len(tool_df)}")
    display(tool_df)

Correct search decisions: 12/12


,message,attempt,should_search,searched,correct,reply
0,hi!,1,False,False,True,"Hello! I'm glad to see you here. If you have any questions about gemstones, their properties, identification, or famous"
1,hi!,2,False,False,True,"Hello! I'm glad to see you here. If you have any questions about gemstones, their properties, identification, or famous"
2,what can you do?,1,False,False,True,"Hello! I'm here to help you with anything related to gemstones. You can ask me about gemstone properties, identification"
3,what can you do?,2,False,False,True,"Hello! I'm here to help you with anything related to gemstones. You can ask me about gemstone properties, identification"
4,thanks (follow-up),1,False,False,True,"You're welcome! If you have more questions about gemstones, like their properties, identification, or famous gems, feel"
5,thanks (follow-up),2,False,False,True,"You're welcome! If you have more questions about gemstones, like their properties, identification, or famous gems, feel"
6,How do I cook carbonara?,1,False,False,True,"I can only help with gemstone-related questions. Try asking me about gemstone properties, identification, or history!"
7,How do I cook carbonara?,2,False,False,True,"I can only help with gemstone-related questions. Try asking me about gemstone properties, identification, or history!"
8,What color is sapphire?,1,True,True,True,"Sapphires are most commonly associated with the color blue, particularly a velvety cornflower blue, which is considered"
9,What color is sapphire?,2,True,True,True,"Sapphires are most commonly associated with the color blue, particularly a velvety cornflower blue, which is considered"


## Part 7: Conclusion

### Results (18 questions, one run each)

| Metric | A: old splitter, k=8 | **B: app chunker, k=8** | B: app chunker, k=4 |
|---|---|---|---|
| Correctness | 0.89 | **0.94** | 0.89 |
| Helpfulness | 0.83 | **1.00** | 0.94 |
| Groundedness (vs retrieved passages) | 0.83 | 0.83 | 0.67 |
| Citation accuracy | 0.92 | **0.96** | 0.98 |
| Retrieval hit | 0.94 | **1.00** | 1.00 |
| Search decision (Part 6, live app) | — | **12/12** | — |

**How much to trust these numbers:** with 18 questions, one question moves an average by about 0.06, and the LLM judge is itself noisy. On the Montana question, A and B gave essentially the same one-sentence answer and the groundedness judge scored one 1 and the other 0. Differences of one question are within noise; the findings below rest on specific answers, not only averages.

### Findings

1. **The structure-aware chunker fixes catalog lookups.** "What is the largest topaz in the Smithsonian collection list?" A didn't retrieve the right passage: its chunk holds catalog rows cut off from the TOPAZ heading, and the bot answered that the sources don't say. B retrieved the labelled catalog chunk and answered "7,725 carats, yellow, Brazil [1]". That's the one retrieval miss in A (0.94 → 1.00) and most of the correctness gain. Fact-sheet questions were answered correctly by both; the old splitter happened to keep feldspar and spinel intact enough at k=8.

2. **B is at least as good everywhere else.** Helpfulness and citation accuracy are higher, groundedness is tied, and no question got worse in a way that held up on inspection. Adopting the chunker is supported; the evidence for a *large* improvement outside the catalog is not.

3. **Keep k=8.** With B at k=4, groundedness dropped to 0.67 and one table question (US diamond import sources) failed. The better chunks don't make the larger k unnecessary.

4. **Biggest remaining problem: outside knowledge on questions the corpus can't answer.** For "What is the hardness of moissanite?", all three configurations said the sources don't cover it, then added "however, it is generally known to be 9.25 on the Mohs scale". That breaks the prompt's rule and is the same pattern as the uncited "boron" claim seen in manual testing. Chunking can't fix it; the system prompt needs to say explicitly not to add "generally known" facts after saying the sources don't cover something. The tanzanite question was handled correctly.

5. **Uncited sentences in longer answers.** In list-style answers (gem cuts, synthetics) gpt-4o-mini sometimes leaves a sentence without a citation and adds detail not in the passages, which is why groundedness is 0.83 rather than higher.

6. **The search decision works.** Greetings, "what can you do?", thanks and an off-topic question were answered without searching; gem questions, including the vague "is my stone good?", always searched (12/12).

### Recommendation

- Merge the structure-aware chunker (B) and keep `topK: 8`.
- Next: tighten the system prompt for unanswerable questions (finding 4), then re-run this notebook. The dataset is reused automatically as long as `test_data` is unchanged.
- To make results firmer: add more catalog and table questions, and run each experiment 2–3 times to average out judge noise.